# 框架图

```mermaid
graph LR
    %% 定义节点样式
    classDef interface fill:#e3f2fd,stroke:#1976d2,stroke-width:3px,color:#000;
    classDef logic fill:#fff3e0,stroke:#f57c00,stroke-width:2px,color:#000;
    classDef utility fill:#e8f5e9,stroke:#388e3c,stroke-width:2px,color:#000;
    classDef callback fill:#f3e5f5,stroke:#7b1fa2,stroke-width:2px,color:#000;
    classDef external fill:#fce4ec,stroke:#c2185b,stroke-width:1px,stroke-dasharray: 3 3,color:#000;

    %% =======================
    %% 外部调用者
    %% =======================
    subgraph External ["外部调用者 (Python/C#/C++)"]
        direction TB
        Ext_User["用户代码<br/>Python/C#/C++"]:::external
    end

    %% =======================
    %% 接口层 (C Export API)
    %% =======================
    subgraph Layer_Interface ["接口层 - External C API"]
        direction TB
        
        subgraph Data_Export ["数据导出 (Binary → CSV)"]
            API_DumpBars["dump_bars<br/>导出K线数据"]:::interface
            API_DumpTicks["dump_ticks<br/>导出Tick数据"]:::interface
        end

        subgraph Data_Import ["数据导入 (CSV → Binary)"]
            API_TransCsv["trans_csv_bars<br/>CSV转二进制K线"]:::interface
        end

        subgraph Data_Read ["数据读取 (Binary → Memory)"]
            API_ReadDSB["read_dsb_bars<br/>read_dsb_ticks<br/>read_dsb_order_details<br/>read_dsb_order_queues<br/>read_dsb_transactions"]:::interface
            API_ReadDMB["read_dmb_bars<br/>read_dmb_ticks"]:::interface
        end

        subgraph Data_Store ["数据存储 (Memory → Binary)"]
            API_Store["store_bars<br/>store_ticks<br/>store_order_details<br/>store_order_queues<br/>store_transactions"]:::interface
        end

        subgraph Data_Process ["高级处理"]
            API_Resample["resample_bars<br/>K线重采样/周期合成"]:::interface
        end
    end

    %% =======================
    %% 核心处理层 (Core Logic)
    %% =======================
    subgraph Layer_Logic ["核心处理层 - Internal Implementation"]
        direction TB
        
        Logic_ProcBlock["proc_block_data<br/>数据块处理<br/>• 解压缩<br/>• 版本转换<br/>• 结构体兼容"]:::logic
        Logic_TimeHelpers["strToDate<br/>strToTime<br/>时间格式转换"]:::logic
    end

    %% =======================
    %% 依赖与基础层 (Dependencies)
    %% =======================
    subgraph Layer_Utils ["依赖与基础工具 - Dependencies"]
        direction TB
        
        Util_Compress["WTSCmpHelper<br/>compress_data<br/>uncompress_data"]:::utility
        Util_FileIO["BoostFile<br/>filesystem<br/>read_file_contents<br/>write_file_contents"]:::utility
        Util_CsvReader["CsvReader<br/>load_from_file<br/>next_row<br/>get_string/get_double"]:::utility
        Util_DataFactory["WTSDataFactory<br/>extractKlineData<br/>K线合成算法"]:::utility
        Util_Json["RapidJSON<br/>Document::Parse<br/>交易时段解析"]:::utility
    end

    %% =======================
    %% 回调机制 (Callbacks)
    %% =======================
    subgraph Mechanisms ["回调机制 - Callbacks"]
        direction TB
        
        CB_Data["数据回调<br/>FuncGetBarsCallback<br/>FuncGetTicksCallback<br/>FuncGetOrdDtlCallback<br/>FuncGetOrdQueCallback<br/>FuncGetTransCallback"]:::callback
        CB_Log["日志回调<br/>FuncLogCallback<br/>进度/错误通知"]:::callback
        CB_Count["计数回调<br/>FuncCountDataCallback<br/>数据总量预告"]:::callback
    end

    %% =======================
    %% 调用关系 - 外部到接口
    %% =======================
    Ext_User -->|调用| API_DumpBars
    Ext_User -->|调用| API_DumpTicks
    Ext_User -->|调用| API_TransCsv
    Ext_User -->|调用| API_ReadDSB
    Ext_User -->|调用| API_ReadDMB
    Ext_User -->|调用| API_Store
    Ext_User -->|调用| API_Resample

    %% =======================
    %% 调用关系 - 导出流程 (dump_bars/dump_ticks)
    %% =======================
    API_DumpBars -->|1. 读取文件| Util_FileIO
    API_DumpBars -->|2. 处理数据块| Logic_ProcBlock
    Logic_ProcBlock -->|解压缩| Util_Compress
    API_DumpBars -->|3. 写入CSV| Util_FileIO
    API_DumpBars -.->|日志通知| CB_Log

    API_DumpTicks -->|1. 读取文件| Util_FileIO
    API_DumpTicks -->|2. 处理数据块| Logic_ProcBlock
    API_DumpTicks -->|3. 写入CSV| Util_FileIO
    API_DumpTicks -.->|日志通知| CB_Log

    %% =======================
    %% 调用关系 - 导入流程 (trans_csv_bars)
    %% =======================
    API_TransCsv -->|1. 读取CSV| Util_CsvReader
    API_TransCsv -->|2. 时间转换| Logic_TimeHelpers
    API_TransCsv -->|3. 压缩数据| Util_Compress
    API_TransCsv -->|4. 写入文件| Util_FileIO
    API_TransCsv -.->|日志通知| CB_Log

    %% =======================
    %% 调用关系 - 读取流程 (read_dsb_*)
    %% =======================
    API_ReadDSB -->|1. 读取文件| Util_FileIO
    API_ReadDSB -->|2. 处理数据块| Logic_ProcBlock
    Logic_ProcBlock -->|解压缩| Util_Compress
    API_ReadDSB -.->|数据回调| CB_Data
    API_ReadDSB -.->|计数回调| CB_Count
    API_ReadDSB -.->|日志回调| CB_Log

    %% =======================
    %% 调用关系 - 读取流程 (read_dmb_*)
    %% =======================
    API_ReadDMB -->|内存映射| Util_FileIO
    API_ReadDMB -.->|直接回调| CB_Data
    API_ReadDMB -.->|计数回调| CB_Count
    API_ReadDMB -.->|日志回调| CB_Log

    %% =======================
    %% 调用关系 - 存储流程 (store_*)
    %% =======================
    API_Store -->|1. 压缩数据| Util_Compress
    API_Store -->|2. 写入文件| Util_FileIO
    API_Store -.->|日志通知| CB_Log

    %% =======================
    %% 调用关系 - 重采样流程 (resample_bars)
    %% =======================
    API_Resample -->|1. 解析交易时段| Util_Json
    API_Resample -->|2. 读取源数据| Util_FileIO
    API_Resample -->|3. 处理数据块| Logic_ProcBlock
    Logic_ProcBlock -->|解压缩| Util_Compress
    API_Resample -->|4. K线合成| Util_DataFactory
    API_Resample -.->|数据回调| CB_Data
    API_Resample -.->|计数回调| CB_Count
    API_Resample -.->|日志回调| CB_Log
```

# proc_block_data
负责对原始数据块进行**标准化处理**，包括自动解压缩和旧版本数据结构的兼容性转换，确保后续逻辑能处理统一格式的数据。
* **检查数据头**
  * 读取数据块头部的标志位。
  * 检查数据是否被压缩 (`is_compressed`)。
  * 检查数据是否为老版本结构 (`is_old_version`)。
  * **快速返回**：如果既未压缩也非老版本，且不需要保留头部，则直接移除头部后返回；否则进入后续处理。
* **数据解压与提取**
  * **解压缩**：如果数据被压缩，根据头部记录的大小，调用 `WTSCmpHelper::uncompress_data` 将数据解压到临时缓冲区 `buffer`。
  * **直接提取**：如果未压缩但需要版本转换，直接将头部之后的数据拷贝到 `buffer`。
* **版本兼容转换**
  * 如果标记为老版本 (`bOldVer`)：
    * **K线数据 (`isBar`)**：遍历老版本结构体 `WTSBarStructOld` 数组，逐个赋值给新版本结构体 `WTSBarStruct`（字段映射），替换缓冲区内容。
    * **Tick数据**：同理，将 `WTSTickStructOld` 转换为 `WTSTickStruct`，替换缓冲区内容。
* **头部处理与重组**
  * **保留头部 (`bKeepHead`)**：
    * 将数据块调整为标准头部大小。
    * 追加处理后的数据 `buffer`。
    * 更新头部版本号为 `BLOCK_VERSION_RAW_V2`（表示已解压的新版数据）。
  * **移除头部**：
    * 直接将处理后的数据 `buffer` 交换给输出变量 `content`（纯数据部分）。
```cpp
/**
 * @brief 处理数据块，包括解压缩和版本转换
 * @param content 数据块内容（输入输出参数，会被修改）
 * @param isBar 是否为K线数据（true=K线数据，false=Tick数据）
 * @param bKeepHead 是否保留数据块头部（true=保留头部，false=移除头部）
 * @return 处理是否成功
 */
bool proc_block_data(std::string& content, bool isBar, bool bKeepHead /* = true */)
```

# 数据导出 (Binary → CSV)

## dump_bars
该函数用于**批量导出 K 线数据**。它遍历指定目录下的二进制存储文件（DSB 格式），将其转换为易读的 CSV 格式文本文件，支持日线和分钟线周期。
* **目录与文件检查**
  * 检查源目录是否存在，若不存在则报错返回。
  * 检查并创建目标 CSV 输出目录。
* **遍历处理文件**
  * 遍历源目录下所有 `.dsb` 后缀的文件。
  * 读取文件全部内容到内存。
  * **头部校验**：检查文件大小是否合法，检查数据类型（`_type`）是否在 K 线范围内（`BT_HIS_Minute1` ~ `BT_HIS_Day`）。
* **数据转换**
  * 调用 `proc_block_data` 对数据进行解压和标准化（不保留头部）。
  * 计算 K 线条数 (`size / sizeof(WTSBarStruct)`).
* **写入 CSV**
  * 构建 CSV 文件名（合约代码.csv）。
  * **写入表头**：`date,time,open,high,low,close,settle,volume,turnover,open_interest,diff_interest`。
  * **遍历 K 线数据**：
    * **日期时间格式化**：
      * 日线：时间字段置 0。
      * 分钟线：解析 `time` 字段，分离出日期（yyyymmdd）和时间（HHMM）。
    * 写入 OHLC、结算价、成交量/额、持仓量等字段。
  * 将生成的字符串内容一次性写入文件。
```cpp
/**
 * @brief 将二进制K线数据导出为CSV格式
 * @param binFolder 二进制数据文件夹路径
 * @param csvFolder CSV输出文件夹路径
 * @param strFilter 文件过滤器（可选）
 * @param cbLogger 日志回调函数（可选）
 */
void dump_bars(WtString binFolder, WtString csvFolder, WtString strFilter = "", FuncLogCallback cbLogger = NULL)

```



## dump_ticks
该函数用于**批量导出 Tick 数据**。它将高频的 Tick 快照二进制数据（DSB 格式）转换为包含完整盘口信息的 CSV 文件。
* **目录与文件检查**
  * 标准化路径，检查源目录存在性，创建目标输出目录。
* **遍历处理文件**
  * 遍历 `.dsb` 文件，读取内容。
  * 校验头部大小是否合法。
* **数据转换**
  * 调用 `proc_block_data`，参数设为 `isBar=false`，解压并标准化 Tick 数据结构。
  * 计算 Tick 条数。
* **构建 CSV 内容**
  * **写入表头**：
    * 基础字段：交易所、代码、时间、价格、OHLC、持仓等。
    * **盘口字段**：循环生成 `bidprice1` ~ `bidprice10`、`bidqty1` ~ `bidqty10` 以及对应的卖盘字段（共 10 档）。
  * **遍历 Tick 数据**：
    * 格式化输出基础行情数据。
    * **盘口循环**：遍历 0-9 索引，写入 10 档买卖价格和挂单量。
  * 将完整内容写入对应的 CSV 文件。
```cpp
/**
 * @brief 将二进制Tick数据导出为CSV格式
 * @param binFolder 二进制数据文件夹路径
 * @param csvFolder CSV输出文件夹路径
 * @param strFilter 文件过滤器（可选）
 * @param cbLogger 日志回调函数（可选）
 */
void dump_ticks(WtString binFolder, WtString csvFolder, WtString strFilter = "", FuncLogCallback cbLogger = NULL)

```

# 数据导入 (CSV → Binary)

## trans_csv_bars

该函数用于**将 CSV 格式的 K 线数据转换为二进制格式**。它遍历指定目录下的 CSV 文件，读取 K 线数据，将其转换为 WonderTrader 标准的压缩二进制（DSB）格式存储，以节省空间并提高读取效率。
* **环境准备与校验**
  * 检查 CSV 源目录 `csvFolder` 是否存在。
  * 检查二进制输出目录 `binFolder`，如果不存在则自动创建。
  * **解析周期**：根据传入的 `period` 参数（"m1", "m5", "d"），确定内部 K 线周期类型（`KP_Minute1`, `KP_Minute5`, `KP_DAY`）。
* **遍历与读取 CSV**
  * 遍历源目录下的所有 `.csv` 文件。
  * 使用 `CsvReader` 加载文件内容。
  * **逐行解析**：
    * 调用 `strToDate` 将日期字符串转换为数值日期。
    * 如果是分钟线，调用 `strToTime` 将时间字符串转换为数值时间，并结合日期计算内部时间格式。
    * 读取 `open`, `high`, `low`, `close`, `volume`, `turnover`, `open_interest`, `diff_interest`, `settle` 等字段。
    * 将解析出的数据存入 `WTSBarStruct` 结构体数组中。
* **压缩与存储**
  * **构建头部**：创建 `HisKlineBlockV2` 结构，设置版本号为 `BLOCK_VERSION_CMP_V2`，并根据周期设置数据类型（如 `BT_HIS_Minute1`）。
  * **数据压缩**：调用 `WTSCmpHelper::compress_data` 对内存中的 K 线数组进行压缩。
  * **写入文件**：
    * 生成对应的 `.dsb` 文件名。
    * 写入文件头。
    * 写入压缩后的数据体。
    * 关闭文件。
```cpp
/**
 * @brief 将CSV格式K线数据转换为二进制格式
 * @param csvFolder CSV数据文件夹路径
 * @param binFolder 二进制输出文件夹路径
 * @param period 数据周期（"m1"=1分钟, "m5"=5分钟, "d"=日线）
 * @param cbLogger 日志回调函数（可选）
 */
void trans_csv_bars(WtString csvFolder, WtString binFolder, WtString period, FuncLogCallback cbLogger /* = NULL */)
```

# 数据读取 (Binary → Memory)

## read_dsb_ticks
该函数用于**读取 DSB 格式的 Tick 数据文件**。它负责从磁盘读取压缩存储的历史 Tick 数据，解压后通过回调函数将数据传递给调用者。
* **读取与校验**
  * 读取指定路径的文件内容到内存缓冲区 `content`。
  * **校验头部**：检查文件大小是否至少包含一个 Tick 数据块头部 (`HisTickBlock`)，若不足则报错返回。
* **数据处理**
  * 调用 `proc_block_data` 对数据进行处理：
    * 自动识别并解压数据（如果是压缩格式）。
    * 兼容性转换（如果是旧版本结构）。
    * `isBar` 设为 `false`（非 K 线数据），`bKeepHead` 设为 `false`（不保留头部，只留纯数据）。
* **计算与回调**
  * **空数据检查**：如果处理后的内容为空，通过 `cbCnt(0)` 通知并返回。
  * **计算条数**：`tcnt = content.size() / sizeof(WTSTickStruct)`。
  * **计数通知**：调用 `cbCnt(tcnt)` 告知即将推送的数据总量。
  * **数据推送**：调用 `cb` 回调函数，将数据指针和数量传递给外部。参数 `isLast` 设为 `true`（一次性推送）。
  * **日志记录**：记录读取完成及数据条数。
```cpp
/**
 * @brief 读取DSB格式的Tick数据文件
 * @param tickFile Tick数据文件路径
 * @param cb Tick数据回调函数
 * @param cbCnt 数据计数回调函数
 * @param cbLogger 日志回调函数（可选）
 * @return 读取的数据条数
 */
WtUInt32 read_dsb_ticks(WtString tickFile, FuncGetTicksCallback cb, FuncCountDataCallback cbCnt, FuncLogCallback cbLogger /* = NULL */)
```

## read_dsb_order_details

该函数用于**读取 DSB 格式的逐笔委托数据**（Level-2 Data）。处理逻辑与读取 Tick 类似，针对的数据结构是 `WTSOrdDtlStruct`。
* **读取与校验**
  * 读取文件内容。
  * 校验文件大小是否满足 `HisOrdDtlBlock` 头部要求。
* **数据处理**
  * 调用 `proc_block_data` 解压和标准化数据（`isBar=false`）。
* **计算与回调**
  * 计算数据条数 (`size / sizeof(WTSOrdDtlStruct)`).
  * 调用 `cbCnt` 通知总量。
  * 调用 `cb` 推送 `WTSOrdDtlStruct` 数组指针。
```cpp
/**
 * @brief 读取DSB格式的逐笔委托数据文件
 * @param dataFile 数据文件路径
 * @param cb 逐笔委托数据回调函数
 * @param cbCnt 数据计数回调函数
 * @param cbLogger 日志回调函数（可选）
 * @return 读取的数据条数
 */
WtUInt32 read_dsb_order_details(WtString dataFile, FuncGetOrdDtlCallback cb, FuncCountDataCallback cbCnt, FuncLogCallback cbLogger/* = NULL*/)
```

## read_dsb_order_queues

该函数用于**读取 DSB 格式的委托队列数据**（Level-2 Data）。处理逻辑与上述函数一致，针对的数据结构是 `WTSOrdQueStruct`。
* **读取与校验**
  * 读取文件内容。
  * 校验文件大小是否满足 `HisOrdQueBlock` 头部要求。
* **数据处理**
  * 调用 `proc_block_data` 解压和标准化数据。
* **计算与回调**
  * 计算数据条数 (`size / sizeof(WTSOrdQueStruct)`).
  * 调用 `cbCnt` 通知总量。
  * 调用 `cb` 推送 `WTSOrdQueStruct` 数组指针。
```cpp
/**
 * @brief 读取DSB格式的委托队列数据文件
 * @param dataFile 数据文件路径
 * @param cb 委托队列数据回调函数
 * @param cbCnt 数据计数回调函数
 * @param cbLogger 日志回调函数（可选）
 * @return 读取的数据条数
 */
WtUInt32 read_dsb_order_queues(WtString dataFile, FuncGetOrdQueCallback cb, FuncCountDataCallback cbCnt, FuncLogCallback cbLogger/* = NULL*/)
```

## read_dsb_transactions

该函数用于**读取 DSB 格式的逐笔成交数据**（Level-2 Data）。针对的数据结构是 `WTSTransStruct`。
* **读取与校验**
  * 读取文件内容。
  * 校验文件大小是否满足 `HisTransBlock` 头部要求。
* **数据处理**
  * 调用 `proc_block_data` 解压和标准化数据。
* **计算与回调**
  * 计算数据条数 (`size / sizeof(WTSTransStruct)`).
  * 调用 `cbCnt` 通知总量。
  * 调用 `cb` 推送 `WTSTransStruct` 数组指针。
```cpp
/**
 * @brief 读取DSB格式的逐笔成交数据文件
 * @param dataFile 数据文件路径
 * @param cb 逐笔成交数据回调函数
 * @param cbCnt 数据计数回调函数
 * @param cbLogger 日志回调函数（可选）
 * @return 读取的数据条数
 */
WtUInt32 read_dsb_transactions(WtString dataFile, FuncGetTransCallback cb, FuncCountDataCallback cbCnt, FuncLogCallback cbLogger/* = NULL*/)
```

## read_dsb_bars

该函数用于**读取 DSB 格式的 K 线数据**。支持日线、分钟线等多种周期，自动处理压缩和解压。
* **读取与校验**
  * 读取文件内容。
  * 校验文件大小是否满足 `HisKlineBlock` 头部要求。
* **数据处理**
  * 调用 `proc_block_data`：
  * 参数 `isBar` 设为 `true`：启用 K 线特有的版本转换逻辑（`WTSBarStructOld` -> `WTSBarStruct`）。
  * `bKeepHead` 设为 `false`。
* **计算与回调**
  * 计算 K 线条数 (`size / sizeof(WTSBarStruct)`).
  * 调用 `cbCnt` 通知总量。
  * 调用 `cb` 推送 `WTSBarStruct` 数组指针。
```cpp
/**
 * @brief 读取DSB格式的K线数据文件
 * @param barFile K线数据文件路径
 * @param cb K线数据回调函数
 * @param cbCnt 数据计数回调函数
 * @param cbLogger 日志回调函数（可选）
 * @return 读取的数据条数
 */
WtUInt32 read_dsb_bars(WtString barFile, FuncGetBarsCallback cb, FuncCountDataCallback cbCnt, FuncLogCallback cbLogger)
```

## read_dmb_bars

该函数用于**读取 DMB 格式（内存映射二进制）的 K 线数据**。DMB 格式通常用于实时数据存储，数据**不压缩**，可以直接映射读取，速度极快。
* **读取与校验**
  * 读取文件内容到缓冲区。
  * 校验文件大小是否满足 `RTKlineBlock` 头部要求。
* **解析与提取**
  * 直接将缓冲区指针强转为 `RTKlineBlock*` 结构体指针。
  * 从头部字段 `_size` 获取 K 线数据条数 `kcnt`。
  * **不进行解压**：DMB 格式设计为非压缩。
* **回调**
  * 调用 `cbCnt` 通知总量。
  * 直接使用 `tBlock->_bars` 指针（变长数组首地址）调用 `cb` 回调，零拷贝传递数据。
```cpp
/**
 * @brief 读取DMB格式的K线数据文件
 * @param barFile K线数据文件路径
 * @param cb K线数据回调函数
 * @param cbCnt 数据计数回调函数
 * @param cbLogger 日志回调函数（可选）
 * @return 读取的数据条数
 */
WtUInt32 read_dmb_bars(WtString barFile, FuncGetBarsCallback cb, FuncCountDataCallback cbCnt, FuncLogCallback cbLogger)
```

## read_dmb_ticks

该函数用于**读取 DMB 格式的 Tick 数据**。同样用于快速读取实时产生的非压缩 Tick 数据。
* **读取与校验**
  * 读取文件内容。
  * 校验文件大小是否满足 `RTTickBlock` 头部要求。
* **解析与提取**
  * 将缓冲区指针强转为 `RTTickBlock*`。
  * 从头部获取 Tick 条数 `tcnt`。
* **回调**
  * 调用 `cbCnt` 通知总量。
  * 直接使用 `tBlock->_ticks` 指针调用 `cb` 回调，传递 Tick 数据数组。
```cpp
/**
 * @brief 读取DMB格式的Tick数据文件
 * @param tickFile Tick数据文件路径
 * @param cb Tick数据回调函数
 * @param cbCnt 数据计数回调函数
 * @param cbLogger 日志回调函数（可选）
 * @return 读取的数据条数
 */
WtUInt32 read_dmb_ticks(WtString tickFile, FuncGetTicksCallback cb, FuncCountDataCallback cbCnt, FuncLogCallback cbLogger /* = NULL */)
```

# 数据存储 (Memory → Binary)

## store_bars

该函数用于**将内存中的 K 线数据存储为二进制文件**。它接收 K 线数据数组，根据指定的周期类型进行封装和压缩，最终保存为 WonderTrader 标准的 DSB 格式文件。
* **参数校验与解析**
  * **数量检查**：检查传入的数据条数 `count`，若为 0 则记录错误日志并返回 `false`。
  * **周期解析**：解析 `period` 参数字符串：
    * `"m1"` -> 对应类型 `BT_HIS_Minute1`
    * `"m5"` -> 对应类型 `BT_HIS_Minute5`
    * `"d"` -> 对应类型 `BT_HIS_Day`
    * 其他 -> 记录错误日志并返回 `false`。
* **数据准备与压缩**
  * **内存拷贝**：将传入的 `firstBar` 数组内容复制到临时缓冲区 `buffer` 中。
  * **构建头部**：创建 `HisKlineBlockV2` 结构体：
    * `_blk_flag`：写入标准文件头标识。
    * `_version`：设为 `BLOCK_VERSION_CMP_V2`（压缩版 V2）。
    * `_type`：设为解析出的 K 线周期类型。
  * **数据压缩**：调用 `WTSCmpHelper::compress_data` 对 K 线数据缓冲区进行压缩，获取压缩后的数据块。
  * **设置大小**：将压缩后的数据长度写入头部的 `_size` 字段。
* **文件写入**
  * **合并内容**：将文件头部与压缩后的数据体拼接成完整的写入内容。
  * **写入磁盘**：
    * 使用 `BoostFile` 创建新文件（如果存在则覆盖）。
    * 将完整内容一次性写入文件。
    * 关闭文件句柄。
```cpp
/**
 * @brief 存储K线数据到二进制文件
 * @param barFile 输出文件路径
 * @param firstBar K线数据数组首地址
 * @param count 数据条数
 * @param period 数据周期（"m1"=1分钟, "m5"=5分钟, "d"=日线）
 * @param cbLogger 日志回调函数（可选）
 * @return 存储是否成功
 */
bool store_bars(WtString barFile, WTSBarStruct* firstBar, int count, WtString period, FuncLogCallback cbLogger = NULL)
```

## store_ticks

该函数用于**将内存中的 Tick 数据存储为二进制文件**。将高频 Tick 快照数据压缩并保存为 DSB 格式。
* **校验**
  * 检查数据条数 `count` 是否大于 0，否则报错返回。
* **数据准备与压缩**
  * **内存拷贝**：将 `firstTick` 数组复制到缓冲区。
  * **构建头部**：创建 `HisTickBlockV2` 结构体：
    * `_version`：`BLOCK_VERSION_CMP_V2`。
    * `_type`：`BT_HIS_Ticks`（Tick 数据类型）。
  * **压缩**：调用 `WTSCmpHelper::compress_data` 对 Tick 数据进行压缩。
  * **设置大小**：记录压缩后的大小。
* **文件写入**
  * 拼接头部和压缩数据。
  * 通过 `BoostFile` 将数据写入指定的 `tickFile` 路径。
```cpp
/**
 * @brief 存储Tick数据到二进制文件
 * @param tickFile 输出文件路径
 * @param firstTick Tick数据数组首地址
 * @param count 数据条数
 * @param cbLogger 日志回调函数（可选）
 * @return 存储是否成功
 */
bool store_ticks(WtString tickFile, WTSTickStruct* firstTick, int count, FuncLogCallback cbLogger = NULL)
```

## store_order_details

该函数用于**存储逐笔委托数据**（Level-2 Data）。主要用于股票等市场的逐笔委托记录持久化。
* **校验**
  * 检查数据条数是否为 0。
* **数据准备与压缩**
  * **内存拷贝**：将 `WTSOrdDtlStruct` 数组复制到缓冲区。
  * **构建头部**：创建 `HisOrdDtlBlockV2` 结构体：
    * `_version`：`BLOCK_VERSION_CMP_V2`。
    * `_type`：`BT_HIS_OrdDetail`（逐笔委托类型）。
  * **压缩**：对数据进行压缩。
  * **设置大小**：记录压缩后的大小。
* **文件写入**
  * 拼接头部和压缩数据，写入目标文件。
```cpp
/**
 * @brief 存储逐笔委托数据到二进制文件
 * @param tickFile 输出文件路径
 * @param firstItem 逐笔委托数据数组首地址
 * @param count 数据条数
 * @param cbLogger 日志回调函数（可选）
 * @return 存储是否成功
 */
bool store_order_details(WtString tickFile, WTSOrdDtlStruct* firstItem, int count, FuncLogCallback cbLogger = NULL)
```

## store_order_queues

该函数用于**存储委托队列数据**（Level-2 Data）。保存买一/卖一等价位上的委托队列详情。
* **校验**
  * 检查数据条数是否为 0。
* **数据准备与压缩**
  * **内存拷贝**：将 `WTSOrdQueStruct` 数组复制到缓冲区。
  * **构建头部**：创建 `HisOrdQueBlockV2` 结构体：
    * `_version`：`BLOCK_VERSION_CMP_V2`。
    * `_type`：`BT_HIS_OrdQueue`（委托队列类型）。
  * **压缩**：对数据进行压缩。
  * **设置大小**：记录压缩后的大小。
* **文件写入**
  * 拼接头部和压缩数据，写入目标文件。
```cpp
/**
 * @brief 存储委托队列数据到二进制文件
 * @param tickFile 输出文件路径
 * @param firstItem 委托队列数据数组首地址
 * @param count 数据条数
 * @param cbLogger 日志回调函数（可选）
 * @return 存储是否成功
 */
bool store_order_queues(WtString tickFile, WTSOrdQueStruct* firstItem, int count, FuncLogCallback cbLogger = NULL)
```

## store_transactions

该函数用于**存储逐笔成交数据**（Level-2 Data）。保存每一笔撮合成交的详细记录。
* **校验**
  * 检查数据条数是否为 0。
* **数据准备与压缩**
  * **内存拷贝**：将 `WTSTransStruct` 数组复制到缓冲区。
  * **构建头部**：创建 `HisTransBlockV2` 结构体：
    * `_version`：`BLOCK_VERSION_CMP_V2`。
    * `_type`：`BT_HIS_Trnsctn`（逐笔成交类型）。
  * **压缩**：对数据进行压缩。
  * **设置大小**：记录压缩后的大小。
* **文件写入**
  * 拼接头部和压缩数据，写入目标文件。
```cpp
/**
 * @brief 存储逐笔成交数据到二进制文件
 * @param tickFile 输出文件路径
 * @param firstItem 逐笔成交数据数组首地址
 * @param count 数据条数
 * @param cbLogger 日志回调函数（可选）
 * @return 存储是否成功
 */
bool store_transactions(WtString tickFile, WTSTransStruct* firstItem, int count, FuncLogCallback cbLogger = NULL)
```

# 高级处理

## resample_bars

该函数用于**K 线数据重采样**。它读取基础周期的二进制 K 线数据（如 1 分钟线），根据指定的交易时段信息，将其合成为更大周期的 K 线数据（如 5 分钟、15 分钟线），并通过回调函数返回结果。
* **参数校验与解析**
  * **解析周期**：识别基础周期 `period`（m1/m5/d）。
  * **时间格式校验**：
    * 如果是日线，要求 `fromTime` 和 `endTime` 为日期格式（yyyymmdd）。
    * 如果是分钟线，要求为时间格式（yyyymmddHHMM）。
    * 确保 `fromTime <= endTime`。
  * **解析交易时段**：解析 JSON 格式的 `sessInfo`，构建 `WTSSessionInfo` 对象，包含时区偏移、集合竞价时间、交易小节等信息，确保重采样时正确处理跨日和休市。
* **读取与定位数据**
  * 读取源 `.dsb` 文件内容到缓冲区。
  * 调用 `proc_block_data` 对数据进行解压和格式标准化。
  * **二分查找**：使用 `std::lower_bound` 在解压后的 K 线数组中，快速定位 `fromTime` 和 `endTime` 对应的起始索引 (`sIdx`) 和结束索引 (`eIdx`)。
* **重采样计算**
  * **构建切片**：根据定位的索引范围，创建 `WTSKlineSlice` 对象，包裹基础数据。
  * **执行合成**：创建 `WTSDataFactory` 工厂对象，调用 `extractKlineData` 方法。
    * 输入：基础 K 线切片、目标倍数 `times`、交易时段信息 `sInfo`。
    * 逻辑：工厂类会根据交易时间段，将 N 根基础 K 线合并为 1 根目标 K 线（计算新的 OHLCV 等）。
* **回调输出**
  * 获取重采样后的数据条数，调用 `cbCnt` 通知调用者。
  * 调用 `cb` 回调函数，将合成后的 `WTSKlineData` 数据指针传递给外部。
  * 释放切片、会话信息和结果数据对象。
```cpp
/**
 * @brief K线数据重采样功能
 * @param barFile 源K线数据文件路径
 * @param cb K线数据回调函数
 * @param cbCnt 数据计数回调函数
 * @param fromTime 开始时间
 * @param endTime 结束时间
 * @param period 基础周期
 * @param times 重采样倍数
 * @param sessInfo 交易时段信息（JSON格式）
 * @param cbLogger 日志回调函数
 * @param bAlignSec 是否按秒对齐
 * @return 重采样后的数据条数
 */
WtUInt32 resample_bars(WtString barFile, FuncGetBarsCallback cb, FuncCountDataCallback cbCnt, 
    WtUInt64 fromTime, WtUInt64 endTime, WtString period, WtUInt32 times, WtString sessInfo, FuncLogCallback cbLogger = NULL, bool bAlignSec = false)
```